# Lab 15: Kaggle Challenge avec MLE-STAR

**Navigation** : [Lab 14 <<](Lab14-Ablation-Refinement.ipynb) | [Index](../../README.md) | [>> Lab 16](../Day7-Production/Lab16-Data-Science-Agent.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Appliquer MLE-STAR à une compétition Kaggle simulée
2. Combiner Web Search + Ablation + Raffinement en workflow complet
3. Générer une soumission compétitive de manière automatisée
4. Itérer sur les améliorations basées sur les résultats

### Prérequis
- Lab 13 et Lab 14 complétés
- Compréhension de MLE-STAR
- Configuration multi-provider active

### Durée estimée : 50-60 minutes

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import json
import re
import numpy as np
import pandas as pd
from typing import List, Dict, Optional
from dataclasses import dataclass
from pathlib import Path

from config import get_settings
from utils import LLMClient
print("Imports OK : json, re, numpy, pandas")

Imports OK : json, re, numpy, pandas


Chargement des parametres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: openrouter


## 2. Kaggle Competition Simulator

In [3]:
@dataclass
class CompetitionInfo:
    name: str
    task: str
    metric: str
    description: str
    data_description: str

class KaggleSimulator:
    def get_competition_info(self) -> CompetitionInfo:
        return CompetitionInfo(
            name='Tabular Playground Series',
            task='binary classification',
            metric='AUC-ROC',
            description='Predict customer churn based on tabular features',
            data_description='20 features numeriques et categoriques, 10000 exemples'
        )

    def generate_sample_data(self, n_samples: int = 500) -> pd.DataFrame:
        np.random.seed(42)
        return pd.DataFrame({
            'customer_id': range(n_samples),
            'age': np.random.randint(18, 80, n_samples),
            'tenure': np.random.randint(1, 72, n_samples),
            'monthly_charges': np.random.uniform(20, 150, n_samples).round(2),
            'total_charges': np.random.uniform(100, 8000, n_samples).round(2),
            'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
            'internet_service': np.random.choice(['DSL', 'Fiber', 'No'], n_samples),
            'target': np.random.randint(0, 2, n_samples)
        })
print("Dataclasses definies : CompetitionInfo, ModelCandidate")

Dataclasses definies : CompetitionInfo, ModelCandidate


## 3. MLE-STAR Agent Complet

In [4]:
class MLEStarAgent:
    def __init__(self):
        self.llm = LLMClient()
        self.competition = None
        self.data = None
        self.code = None

    def understand_competition(self, info: CompetitionInfo) -> str:
        print('[UNDERSTAND] Analyse de la competition...')
        prompt = f"""Analyse cette competition Kaggle:

NOM: {info.name}
TACHE: {info.task}
METRIQUE: {info.metric}

Resume en 2-3 phrases."""
        response = self.llm.generate(prompt, temperature=0.3)
        return response

    def search_sota(self, task: str) -> List[str]:
        print('[SEARCH] Recherche SOTA...')
        prompt = f"""Pour la tache '{task}', quels sont les 3 modeles les plus performants?
Donne juste les noms, un par ligne."""
        response = self.llm.generate(prompt, temperature=0.3)
        models = re.findall(r'\d+\.\s*(.+)', response)
        return models[:3] if models else ['RandomForest', 'XGBoost', 'LightGBM']

    def generate_initial_code(self, info: CompetitionInfo, models: List[str]) -> str:
        print('[CODE] Generation du code initial...')
        prompt = f"""Genere un script Python pour cette competition.
TACHE: {info.task}
METRIQUE: {info.metric}
MODELES: {', '.join(models[:2])}

Le script doit charger les donnees, entrainer un modele et evaluer.
```python
[code]
```"""
        response = self.llm.generate(prompt, temperature=0.3)
        match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
        return match.group(1).strip() if match else '# Code generation failed'

    def run_pipeline(self, info: CompetitionInfo) -> Dict:
        print('='*50)
        print('MLE-STAR PIPELINE')
        print('='*50)
        understanding = self.understand_competition(info)
        models = self.search_sota(info.task)
        self.code = self.generate_initial_code(info, models)
        return {'understanding': understanding, 'models': models, 'code': self.code}
print("Classe MLEStarAgent definie")

Classe MLEStarAgent definie


## 4. Test du Pipeline

In [5]:
# Initialiser le simulateur
simulator = KaggleSimulator()
info = simulator.get_competition_info()
sample_data = simulator.generate_sample_data(300)

print(f'Competition: {info.name}')
print(f'Data shape: {sample_data.shape}')

Competition: Tabular Playground Series
Data shape: (300, 8)


Execution du pipeline MLE-STAR complet.

In [6]:
# Executer le pipeline MLE-STAR
agent = MLEStarAgent()
result = agent.run_pipeline(info)

MLE-STAR PIPELINE
[UNDERSTAND] Analyse de la competition...


[SEARCH] Recherche SOTA...


[CODE] Generation du code initial...


## 5. Affichage des Resultats

In [7]:
print('\\n' + '='*50)
print('COMPREHENSION:')
print('='*50)
print(result['understanding'][:400])

\n==================================================
COMPREHENSION:
La compétition Kaggle "Tabular Playground Series" est une tâche de classification binaire où l'objectif est de prédire une variable cible à partir de données tabulaires. La performance des modèles est évaluée via la métrique AUC-ROC, qui mesure la capacité du modèle à distinguer entre les deux classes. Cette compétition est souvent utilisée pour expérimenter différentes techniques de machine learn


Synthese des metriques d'amelioration.

In [8]:
print('\\n' + '='*50)
print('MODELES SOTA:')
print('='*50)
for m in result['models']:
    print(f'  - {m}')

\n==================================================
MODELES SOTA:
  - Logistic Regression  
  - Random Forest  
  - Gradient Boosting (e.g., XGBoost)


Affichage des resultats detailles du challenge.

In [9]:
print('\\n' + '='*50)
print('CODE GENERE:')
print('='*50)
print(result['code'][:500] + '...' if len(result['code']) > 500 else result['code'])

\n==================================================
CODE GENERE:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Charger les données
# Remplacez 'data.csv' par le chemin vers votre fichier de données
data = pd.read_csv('data.csv')

# Supposons que la colonne cible s'appelle 'target'
X = data.drop(columns=['target'])
y = data['target']

# Séparer en données d'entraînement et de test
X_trai...


## 6. Resume du Lab
### Workflow MLE-STAR
1. Understand: Analyser la competition
2. Search: Trouver les modeles SOTA
3. Code: Generer une solution initiale
4. Ablation: Identifier les ameliorations
5. Refine: Iterer
### Resultats MLE-Bench
MLE-STAR obtient 63.6% de medailles sur MLE-Bench-Lite.
### Prochaine etape
Lab 16: Data Science Agent avec GCP

## Exemple guide : Compet Kaggle Simule

Creez un workflow MLE-STAR complet pour une competition de votre choix.

### Objectifs
1. Choisir une competition type (classification, regression, NLP, etc.)
2. Executer le pipeline complet
3. Analyser les resultats et suggerer des ameliorations
4. Implementer une iteration de raffinement

### Instructions



In [10]:
#Exemple guide: Definissez votre competition personnalisee
ma_competition = CompetitionInfo(
    name='Ma Competition',
    task='...',  # Ex: 'image classification', 'sentiment analysis'
    metric='...',  # Ex: 'F1-score', 'RMSE'
    description='...',
    data_description='...'
)

#Exemple guide: Generez des donnees synthetiques appropriees
def generate_my_data(n_samples=500):
    np.random.seed(42)
    # Creez un DataFrame adapte a votre tache
    df = pd.DataFrame({
        # Vos features ici
        'target': np.random.randint(0, 2, n_samples)  # Adaptez selon la tache
    })
    return df

#Exemple guide: Executez le pipeline MLE-STAR
agent = MLEStarAgent()
result = agent.run_pipeline(ma_competition)

#Exemple guide: Analysez le code genere et proposez une amelioration specifique
# Exemple: ajouter du feature engineering, changer de modele, etc.
amelioration_proposee = """
# Votre amelioration ici
"""

#Exemple guide: (Bonus) Implementez un cycle d'ablation + raffinement

MLE-STAR PIPELINE
[UNDERSTAND] Analyse de la competition...


[SEARCH] Recherche SOTA...


[CODE] Generation du code initial...


### Extensions
- Integrez la recherche web reelle pour trouver les modeles SOTA
- Ajoutez une evaluation sur donnees de validation
- Simulez plusieurs iterations avec amelioration du score


## Conclusion

Ce notebook a permis d'explorer les aspects essentiels de lab15 kaggle challenge. Les points cles :

- Les concepts fondamentaux ont ete presentes et illustres
- Les exercices proposent une mise en pratique progressive
- Les resultats obtenus permettent de valider la comprehension

**Pour aller plus loin** : approfondir les aspects avances du sujet et explorer les liens avec d'autres domaines.